# AI4Sustain : Classifier Evaluation Demo

**Course:** LLM Spring 2026 &nbsp;|&nbsp; **Dataset:** 101 hand-labeled sustainability news articles &nbsp;|&nbsp; **Author:** Duhita Pradhan

This notebook evaluates three approaches to classifying environmental news articles into five sustainability categories: `renewable`, `emissions`, `biodiversity`, `water`, and `policy`.

| # | Classifier | Approach |
|---|-----------|----------|
| 1 | Keyword Baseline | Rule-based keyword matching |
| 2 | GPT-4o-mini Zero-Shot | LLM prompting, no fine-tuning |
| 3 | DeBERTa-v3 NLI | Zero-shot Natural Language Inference |

It also demonstrates **G-Eval** : an LLM-as-judge technique that scores AI-generated summaries on relevance, coherence, and grounding.

## Setup

Uncomment the install line if running in a fresh environment. Export `OPENAI_API_KEY` in your shell before running the GPT and G-Eval cells; without it those cells fall back to pre-computed results from `data/eval_results.json`.

In [ ]:
# !pip install openai pandas

In [ ]:
import json
import os
import re
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI

CATEGORIES = ['renewable', 'emissions', 'biodiversity', 'water', 'policy']

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
DATA_DIR = Path('data')

---
## 1. Load & Explore the Dataset

`labeled_articles.json` is a hand-curated test set of 101 sustainability news articles. Each record contains four fields:

| Field | Description |
|-------|-------------|
| `title` | Article headline |
| `snippet` | 1–2 sentence excerpt |
| `source` | News outlet |
| `label` | Ground-truth category |

The label distribution is roughly balanced (18–28 articles per category), which makes macro-averaged F1 a meaningful summary statistic.

In [ ]:
with open(DATA_DIR / 'labeled_articles.json') as f:
    articles = json.load(f)

df = pd.DataFrame(articles)
print(f'Total articles: {len(df)}\n')

print('Label distribution:')
dist = df['label'].value_counts().rename_axis('category').reset_index(name='count')
print(dist.to_string(index=False))

print()
print('Sample article:')
print(json.dumps(articles[0], indent=2))

---
## 2. Keyword Baseline Classifier

The keyword baseline counts how many words from each category's vocabulary appear in the combined title and snippet, then assigns the category with the highest count. It is fully deterministic — no model weights, no API calls — and serves as the performance floor.

**Known limitations:**
- Defaults to `renewable` when no keywords match, inflating its recall at the cost of precision.
- Misses synonyms and technical jargon not in the lists (e.g., a `policy` article on solar subsidies scores higher on `renewable`).
- Multi-word phrases like `sea level` require substring matching, not tokenization, so partial matches can occur.

In [ ]:
KEYWORD_LISTS = {
    'renewable':    ['solar', 'wind', 'renewable', 'geothermal', 'hydrogen',
                     'turbine', 'photovoltaic', 'battery', 'ev', 'electric vehicle'],
    'emissions':    ['carbon', 'emission', 'methane', 'co2', 'greenhouse',
                     'fossil', 'net zero', 'decarbonize', 'coal'],
    'biodiversity': ['species', 'forest', 'wildlife', 'ecosystem', 'biodiversity',
                     'deforestation', 'coral', 'reef', 'extinction'],
    'water':        ['ocean', 'water', 'flood', 'drought', 'river',
                     'sea level', 'groundwater', 'glacier', 'aquifer'],
    'policy':       ['policy', 'agreement', 'cop', 'law', 'regulation',
                     'government', 'treaty', 'pledge', 'summit', 'legislation'],
}


def classify_keyword(title, snippet=''):
    text = (title + ' ' + snippet).lower()
    best, best_score = 'renewable', 0
    for cat in CATEGORIES:
        score = sum(1 for w in KEYWORD_LISTS[cat] if w in text)
        if score > best_score:
            best_score, best = score, cat
    return best


def compute_metrics(gold, pred):
    total   = len(gold)
    correct = sum(g == p for g, p in zip(gold, pred))
    per_class = {}
    for cat in CATEGORIES:
        tp  = sum(g == cat and p == cat for g, p in zip(gold, pred))
        fp  = sum(g != cat and p == cat for g, p in zip(gold, pred))
        fn  = sum(g == cat and p != cat for g, p in zip(gold, pred))
        pr  = tp / (tp + fp) if tp + fp > 0 else 0.0
        rec = tp / (tp + fn) if tp + fn > 0 else 0.0
        f1  = 2 * pr * rec / (pr + rec) if pr + rec > 0 else 0.0
        per_class[cat] = {
            'precision': round(pr,  3),
            'recall':    round(rec, 3),
            'f1':        round(f1,  3),
            'support':   gold.count(cat),
        }
    macro_f1 = round(sum(per_class[c]['f1'] for c in CATEGORIES) / len(CATEGORIES), 3)
    return {'accuracy': round(correct / total, 3), 'macroF1': macro_f1, 'perClass': per_class}

In [ ]:
gold       = [a['label'] for a in articles]
kw_preds   = [classify_keyword(a['title'], a.get('snippet', '')) for a in articles]
kw_metrics = compute_metrics(gold, kw_preds)

acc, f1 = kw_metrics['accuracy'], kw_metrics['macroF1']
print(f'Keyword Baseline  :  Accuracy: {acc:.1%}   Macro F1: {f1:.3f}\n')
kw_df = pd.DataFrame(kw_metrics['perClass']).T[['precision', 'recall', 'f1', 'support']]
print(kw_df.to_string())

---
## 3. GPT-4o-mini Zero-Shot Classifier

Instead of keyword lists, GPT-4o-mini reads each article title and assigns a label based on its language understanding — with no task-specific training. Articles are sent in batches of 20 to keep API calls efficient; the model is prompted to return a plain JSON array of labels.

If a batch fails to parse or returns an invalid label, the classifier falls back to the keyword baseline for that article — matching the behaviour of `server/classifier.js`.

> **Requires `OPENAI_API_KEY`.** If the key is absent, pre-computed results are loaded from `data/eval_results.json`.

In [ ]:
def gpt_classify_batch(batch, client):
    n = len(batch)
    lines = []
    for i, a in enumerate(batch):
        lines.append(f'[{i}] ' + a['title'])
    prompt = (
        f'Classify EXACTLY {n} articles.\n'
        f'Return EXACTLY {n} strings in a JSON array.\n'
        'Categories: renewable, emissions, biodiversity, water, policy\n\n'
        + '\n'.join(lines)
        + '\n\nReturn only the JSON array, example: ["renewable","policy","water"]'
    )
    resp  = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=300,
        temperature=0,
    )
    raw   = resp.choices[0].message.content.strip()
    clean = re.sub(r'```(?:json)?', '', raw).replace('```', '').strip()
    try:
        preds = json.loads(clean)
        if not isinstance(preds, list):
            raise ValueError
        result = []
        for i, p in enumerate(preds[:n]):
            fallback = classify_keyword(batch[i]['title'], batch[i].get('snippet', ''))
            result.append(p if p in CATEGORIES else fallback)
        return result
    except Exception:
        return [classify_keyword(a['title'], a.get('snippet', '')) for a in batch]


def gpt_classify_all(articles, api_key):
    client    = OpenAI(api_key=api_key)
    BATCH     = 20
    all_preds = []
    for i in range(0, len(articles), BATCH):
        batch = articles[i:i + BATCH]
        all_preds.extend(gpt_classify_batch(batch, client))
        if i + BATCH < len(articles):
            time.sleep(0.5)
    return all_preds

In [ ]:
if OPENAI_API_KEY:
    print('Running GPT-4o-mini on all 101 articles (6 batches of 20)...')
    gpt_preds   = gpt_classify_all(articles, OPENAI_API_KEY)
    gpt_metrics = compute_metrics(gold, gpt_preds)
else:
    print('OPENAI_API_KEY not set : loading pre-computed GPT results from eval_results.json')
    with open(DATA_DIR / 'eval_results.json') as f:
        _er = json.load(f)
    gpt_metrics = _er['gpt']

acc, f1 = gpt_metrics['accuracy'], gpt_metrics['macroF1']
print(f'\nGPT-4o-mini Zero-Shot  :  Accuracy: {acc:.1%}   Macro F1: {f1:.3f}\n')
gpt_df = pd.DataFrame(gpt_metrics['perClass']).T[['precision', 'recall', 'f1', 'support']]
print(gpt_df.to_string())

---
## 4. DeBERTa-v3 Zero-Shot NLI (Pre-computed)

`cross-encoder/nli-deberta-v3-small` frames classification as **Natural Language Inference**: for each article it scores how strongly each category description *entails* the article text, then picks the highest-scoring label.

| Short key | NLI hypothesis |
|-----------|---------------|
| `renewable` | renewable energy |
| `emissions` | carbon emissions |
| `biodiversity` | biodiversity |
| `water` | water resources |
| `policy` | climate policy |

Running DeBERTa requires ~500 MB of model weights and takes 2–3 minutes on CPU. The backend (`server/eval_runner.js`) runs it once at startup and caches the results in `data/eval_results.json`. We load those results here rather than re-running.

In [ ]:
with open(DATA_DIR / 'eval_results.json') as f:
    eval_results = json.load(f)

deberta_metrics = eval_results['deberta']

acc, f1 = deberta_metrics['accuracy'], deberta_metrics['macroF1']
print(f'DeBERTa-v3 NLI  :  Accuracy: {acc:.1%}   Macro F1: {f1:.3f}\n')
deberta_df = pd.DataFrame(deberta_metrics['perClass']).T[['precision', 'recall', 'f1', 'support']]
print(deberta_df.to_string())

---
## 5. Classifier Comparison

The table below summarises all three classifiers on the full 101-article test set. Per-class F1 scores reveal where each approach succeeds and where it struggles.

In [ ]:
rows = []
for name, m in [
    ('Keyword Baseline',      kw_metrics),
    ('GPT-4o-mini Zero-Shot', gpt_metrics),
    ('DeBERTa-v3 NLI',        deberta_metrics),
]:
    row = {
        'Classifier': name,
        'Accuracy':   m['accuracy'],
        'Macro F1':   m['macroF1'],
    }
    for c in CATEGORIES:
        row[f'F1({c[:3]})'] = m['perClass'][c]['f1']
    rows.append(row)

cmp = pd.DataFrame(rows).set_index('Classifier')
print(cmp.to_string())

**Key observations:**

- **DeBERTa-v3 NLI** achieves the best macro F1 (0.607), with strong per-class scores on `renewable` (0.737) and `water` (0.723).
- **Keyword Baseline** is competitive overall (macro F1 = 0.527) and actually wins on `water` (0.765) — a category with a highly distinctive vocabulary.
- **GPT-4o-mini Zero-Shot** is the most balanced across categories (no single category collapses) but scores lowest overall (0.515), likely because short article titles alone provide insufficient context for confident label assignment.
- All three classifiers struggle most with `policy`, which frequently co-occurs with vocabulary from other categories (e.g., solar subsidy legislation scores high on both `renewable` and `policy`).

---
## 6. G-Eval: LLM-as-Judge Summary Scoring

**G-Eval** uses an LLM as an automated evaluator. Given a set of source articles and an AI-generated summary, GPT-4o-mini scores the summary on three quality dimensions:

| Dimension | Definition |
|-----------|------------|
| **Relevance** | Does the summary capture the key themes of the source articles? |
| **Coherence** | Is the summary well-structured and easy to read? |
| **Grounding** | Are the summary's claims directly supported by the source text? |

Scores range from **1.0** (poor) to **5.0** (excellent).

### Workflow

1. Select a sample of articles.
2. Ask GPT-4o-mini to write a 3–4 sentence summary.
3. Send the summary and article titles back to GPT-4o-mini as a judge and request JSON scores.

This mirrors the `gEval()` function in `server/geval.js` and the evaluation pipeline in `server/eval_runner.js`.

In [ ]:
SAMPLE = articles[:5]

print('Articles used for summary generation:\n')
for i, a in enumerate(SAMPLE, 1):
    label = a['label']
    title = a['title']
    print(f'  {i}. [{label:>12}]  {title}')

In [ ]:
def g_eval(summary, article_titles, client):
    titles_str = '; '.join(article_titles[:10])
    prompt = (
        'You are evaluating an AI-generated news summary.\n'
        'Source article titles: ' + titles_str + '\n'
        'Summary: ' + summary + '\n\n'
        'Rate on three dimensions from 1.0 to 5.0.\n'
        'Return ONLY valid JSON, no markdown fences:\n'
        '{"relevance": X.X, "coherence": X.X, "grounding": X.X}'
    )
    resp  = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=50,
        temperature=0,
    )
    raw   = resp.choices[0].message.content.strip()
    clean = re.sub(r'```(?:json)?', '', raw).replace('```', '').strip()
    scores = json.loads(clean)
    return {k: round(float(v), 2) for k, v in scores.items()}


if OPENAI_API_KEY:
    client = OpenAI(api_key=OPENAI_API_KEY)

    sample_lines = []
    for i, a in enumerate(SAMPLE):
        snippet = a.get('snippet', '')[:80]
        sample_lines.append(f'{i+1}. {a["title"]} : {snippet}')
    sample_text = '\n'.join(sample_lines)

    sum_resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content':
            'Summarize these sustainability news articles in 3-4 sentences:\n' + sample_text
        }],
        max_tokens=200,
        temperature=0.3,
    )
    generated_summary = sum_resp.choices[0].message.content.strip()
    geval_scores = g_eval(generated_summary, [a['title'] for a in SAMPLE], client)

else:
    # Pre-computed from the backend eval run stored in data/eval_results.json
    generated_summary = (
        'In 2024 and early 2025, Southeast Asia recorded over 45 GW of new solar capacity '
        'while EU carbon prices surged past 70 euros per tonne under tighter ETS caps. '
        'Brazil achieved a 50% drop in Amazon deforestation, though ocean temperatures hit '
        'their second consecutive record high, threatening coral reefs and marine food webs. '
        'Ahead of COP30 in Belem, Brazil pledged to end illegal deforestation and cut methane '
        'emissions by 30% by 2030.'
    )
    geval_scores = eval_results['geval']

print('Generated summary:\n')
print(generated_summary)
print()
print('G-Eval scores (out of 5.0):\n')
for dim, score in geval_scores.items():
    filled = int(round(float(score)))
    bar    = '\u2588' * filled + '\u2591' * (5 - filled)
    print(f'  {dim:<12}  {bar}  {score}')

**Interpretation:** Scores above 4.0 indicate high quality. The summary scores well on all three dimensions, reflecting that GPT-4o-mini produces coherent, grounded summaries when the source articles are short and topically clear.

G-Eval is most valuable as a **regression signal**: a sudden drop in coherence or grounding after a prompt or model change indicates quality degradation before any human reviewer sees the output. In production, `server/geval.js` runs this automatically in the background after every live analysis request.